# Colab Training Workflow

Use this notebook to mount Drive, fetch or upload the RDD dataset, convert it to an ImageFolder layout, train the baseline classifier, and save artifacts for reuse.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 3: Locate the extracted RDD2022 dataset and use it as the source root
from pathlib import Path

repo_root = Path('/content/HackKU-2026')
archive_dir = repo_root / 'archive'
archive_dir.mkdir(parents=True, exist_ok=True)

drive_root = Path('/content/drive/MyDrive')
content_root = Path('/content')

def is_rdd_split_root(path: Path):
    return (
        path.is_dir()
        and (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'labels').is_dir()
        and (path / 'val' / 'images').is_dir()
        and (path / 'val' / 'labels').is_dir()
    )

def find_rdd_split_candidates(base: Path):
    candidates = []
    if is_rdd_split_root(base):
        candidates.append(base)
    for split_dir in base.rglob('RDD_SPLIT'):
        if is_rdd_split_root(split_dir):
            candidates.append(split_dir)
    return candidates

search_roots = [repo_root / 'archive', drive_root, content_root]
dataset_candidates = []
for root in search_roots:
    if root.exists():
        dataset_candidates.extend(find_rdd_split_candidates(root))

dataset_candidates = list(dict.fromkeys(dataset_candidates))

if not dataset_candidates:
    raise FileNotFoundError(
        'Could not find an extracted RDD_SPLIT folder in the cloned repo, Drive, or /content. '
        'Place the extracted dataset so it contains train/val/test folders with images and labels.'
    )

dataset_root = dataset_candidates[0]
print('Using dataset root:', dataset_root)
print('Top-level entries:')
for p in sorted(dataset_root.iterdir()):
    print(' -', p.name)

In [ ]:
# Cell 4: Convert the extracted RDD split folders to ImageFolder classes for classifier training
%cd /content/HackKU-2026
!python prepare_rdd_imagefolder.py --source "{dataset_root}" --output data/rdd_imagefolder --splits train val --mode copy --include-empty

In [ ]:
# Cell 5: Train baseline classifier and save artifacts
%cd /content/HackKU-2026
!python train_baseline.py --data_dir data/rdd_imagefolder/train --epochs 5 --batch_size 32 --output_dir artifacts

In [ ]:
# Cell 6: Verify artifacts and preview validation metrics
import json
from pathlib import Path

artifacts = Path('/content/HackKU-2026/artifacts')
print('Artifacts directory:', artifacts)
!ls -lah /content/HackKU-2026/artifacts

report_path = artifacts / 'val_report.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('\nPer-class precision/recall/f1:')
    for label, stats in report.items():
        if isinstance(stats, dict) and 'f1-score' in stats:
            print(f"{label:30s}  p={stats['precision']:.3f}  r={stats['recall']:.3f}  f1={stats['f1-score']:.3f}")
else:
    print('val_report.json not found yet.')

In [ ]:
# Cell 7: Copy artifacts to Google Drive for reuse
from pathlib import Path

src = Path('/content/HackKU-2026/artifacts')
dst = Path('/content/drive/MyDrive/HackKU_artifacts')
dst.mkdir(parents=True, exist_ok=True)

files = ['baseline_model.pt', 'labels.json', 'val_report.json']
for name in files:
    src_file = src / name
    if src_file.exists():
        !cp -f "{src_file}" "{dst / name}"
    else:
        print('Missing:', src_file)

print('Saved artifacts to:', dst)
!ls -lah /content/drive/MyDrive/HackKU_artifacts

In [ ]:
# Cell 8: Sync this notebook file to Google Drive if it is available in the runtime
from pathlib import Path
import shutil

notebook_name = 'train_dataset.ipynb'
drive_notebook_dir = Path('/content/drive/MyDrive/HackKU_notebooks')
drive_notebook_dir.mkdir(parents=True, exist_ok=True)
target_notebook = drive_notebook_dir / notebook_name

candidate_sources = [
    Path('/content') / notebook_name,
    Path('/content/drive/MyDrive') / notebook_name,
    Path('/content/drive/MyDrive/HackKU_notebooks') / notebook_name,
]

for source in candidate_sources:
    if source.exists() and source.resolve() != target_notebook.resolve():
        shutil.copy2(source, target_notebook)
        print('Synced notebook to:', target_notebook)
        break
else:
    if target_notebook.exists():
        print('Notebook already present in Drive at:', target_notebook)
    else:
        print('Notebook file not available in the runtime to sync automatically.')
        print('Use Drive > Save a copy in Drive if you opened the notebook directly from Colab.')